# ITS DT

In [2]:
# Load library
import geopandas as gpd
import pandas as pd
import osmnx as ox
import rasterio
from rasterstats import zonal_stats

In [3]:
# Query campus boundary
boundary = ox.geocode_to_gdf("Institut Teknologi Sepuluh Nopember")

In [4]:
# Query 2D buildings
buildings = ox.features_from_polygon(
    boundary.iloc[0]["geometry"], tags={"building": True}
)

# Remove buildings tagged as point or multipoint
buildings = buildings[
    ~buildings["geometry"].geom_type.isin(["Point", "MultiPoint"])
].reset_index()

# Retain only few attributes for smaller storage
buildings = buildings[["id", "geometry", "building", "name", "building:levels", "height"]]

In [31]:
# Extract median building height from Open Buildings 2.5D
def assign_median_heights(
    building_gdf, raster_path, nodata_val: float = 0.0
):
    """
    Extracts the median raster value for each building footprint.
    """
    # Match CRS
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    if building_gdf.crs != raster_crs:
        building_gdf = building_gdf.to_crs(raster_crs)

    # Compute zonal statistics
    stats = zonal_stats(
        vectors=building_gdf, raster=raster_path, stats=["median"], nodata=nodata_val
    )

    # Assign to GeoDataFrame
    building_gdf["median_height"] = [stat["median"] for stat in stats]

    # Fallback for very small polygons that missed pixel centroids
    building_gdf["median_height"] = building_gdf["median_height"].fillna(0.0)

    # Rounded to 2 decimals
    building_gdf["median_height"] = building_gdf["median_height"].round(2)
    
    # Get area
    building_gdf["area"] = building_gdf.area.round(2)

    return building_gdf

buildings = assign_median_heights(
    buildings, raster_path="../data/building_height_raster_2023.tif"
)

In [35]:
# Export
buildings.to_parquet("../data/buildings_w_height.parquet")

In [5]:
# Re-read
buildings = gpd.read_parquet("../data/buildings_w_height.parquet")

In [8]:
buildings["area"].sum()/(30*30)

np.float64(345.5828)